# Load and analyse the data
In this notebook we will explore the different ways to load the data, visualize it, and make a preliminary analysis.

In [1]:
import sys
sys.path.append("/Users/samuel/Código/TFG")

In [2]:
import librosa as lib
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import src.io
import src.datasets.hls
import src.datasets.nih
import src.inventory
from pathlib import Path
import soundfile as sf
import hashlib
import re

## HLS-CMDS Data

In [ ]:
#Load annotations for HLS-CMDS
dataDir = "../data/raw/HLS-CMDS"
#Load annotations for HLS-CMDS
lung = pd.read_csv(f"{dataDir}/LS.csv")

#Load lung sounds into array where each element is one file
lungFiles = sorted((i for i in os.listdir(f"{dataDir}/LS") if i.endswith(".wav")))
lungSounds = np.empty(len(lungFiles), dtype=object)
lungSR = np.empty(len(lungFiles), dtype=object)
for n, i in enumerate(lungFiles):
    f, sr = lib.load(f"{dataDir}/LS/{i}", sr=None)
    lungSounds[n] = f
    lungSR[n] = sr
print(type(lungFiles))
print(type(lungSounds))
print(type(lungSR))

<class 'list'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [ ]:
for n, f in enumerate(lungSounds):
        fig, ax = plt.subplots(ncols=4, figsize=(20,4))
        nffts = [256, 512, 1024, 2048]
        for i in range(len(nffts)): 
            #Transform the amplitude to dB so we have a logarithmic instead of linear plot
            spec = lib.amplitude_to_db(np.abs(lib.stft(f, n_fft=nffts[i])), ref=np.max)
            img = lib.display.specshow(spec, ax=ax[i], sr=lungSR[n], hop_length=nffts[i] // 4,
                                    x_axis="time", y_axis="linear")
            ax[i].set_title(f"{lungFiles[n]}, n_ffts={nffts[i]}, sr={lungSR[n]}")

### Con esta visualización se concluye que el valor óptimo para n_fft es 256, pues es aquel con el que la imagen queda más definida

## NIH Data

In [3]:
#Load annotations for HLS-CMDS
dataDirNIH = "../data/raw/NIH"
lungNIH, lungFilesNIH, lungSoundsNIH, lungSRNIH = src.io.load_dataset_xlsx(dataDirNIH, "Data annotation.xlsx", "Audio Files")

lungNIH

,Age,Gender,Location,Sound type,Diagnosis
0,70,M,P L L,I E W,Asthma
1,52,F,P L L,E W,Asthma
2,50,F,P L L,I E W,Asthma
3,72,F,P R L,I C,Heart Failure + Lung Fibrosis
4,71,M,P R L,I C E W,Heart Failure + COPD
...,...,...,...,...,...
107,63,M,P R L,E W,COPD
108,26,M,P L M,N,N
109,62,M,P L L,E W,COPD
110,51,M,P R L,E W,COPD


# Create inventory and annotations file(s)

## Inventory

In [3]:
inv = src.inventory.generate()
inv

Inventory file has been generated


,file,dataset,patient,position,hash,sample_rate,channels,bit_depth,is_float,peak,rms,dc_offset,clipping,duration,annotation
0,L0014.wav,HLS-CMDS,HLS-Mix13,LUA,6e16fbcfbcae5accbba8bcd344b617e3e20ade4749f727...,4000,1,16,False,0.034668,0.005713,-1.188151e-06,0.000000,15.000,None
1,L0028.wav,HLS-CMDS,HLS-Mix27,RC,ae5100abccff274c0aa3b159fbbec03741a3405774b404...,4000,1,16,False,0.046295,0.005200,3.433736e-06,0.000000,15.000,None
2,M0005.wav,HLS-CMDS,HLS-Mix4,RUSB,ab5b5fa88f88471147005fffb0d8a400040098d9eb55f1...,4000,1,16,False,0.091583,0.006620,7.553101e-07,0.000000,15.000,None
3,H0140.wav,HLS-CMDS,HLS-Mix139,RLA,8dcaf616ccb1aa74674c7a29f6d16e5ee91c0d5e3b2af1...,4000,1,16,False,0.198608,0.007852,1.919047e-06,0.000000,15.000,None
4,M0011.wav,HLS-CMDS,HLS-Mix10,Apex,f67c2e00075f912635272294c9ad195a282876b8f0ef35...,4000,1,16,False,0.019775,0.003763,3.550212e-07,0.000000,15.000,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
866,"DP79_asthma,E W,A R U,46,M.wav",NIH,NIH79,A R U,cf22c72a9ec1c7a69a9a9003eb0689b5637fe58d8b347a...,4000,1,16,False,0.999969,0.074263,-4.895800e-03,0.000012,20.336,None
867,"DP69_pneumonia,Bronchial,P R L ,64,M.wav",NIH,NIH69,P R L,f52208b3dc08e6ca908c425c57d1272312c864fac15195...,4000,1,16,False,1.000000,0.064400,1.221145e-03,0.000010,24.176,None
868,"BP13_Heart Failure,C,P L L,55,M.wav",NIH,NIH13,P L L,3cdd5ad5094809551f58b129207c4b5e31fae066d2e2a6...,4000,1,16,False,1.000000,0.174402,-1.121447e-04,0.000051,14.576,None
869,"DP107_Asthma,E W,P L U,59,F.wav",NIH,NIH107,P L U,90f55d4ff96dc2fed44ae5f947a0066f42ef6c8630f37a...,4000,1,16,False,0.999969,0.105253,2.235783e-03,0.000040,12.656,None
